## Setup

In [1]:
import requests
import os

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
response = requests.get(url)

if response.status_code == 200:
  with open("Chinook.db" , "wb") as file:
    file.write(response.content)

else:
  print(f"Failed to download the file. Status code :{response.status_code}")

In [2]:
from langchain_ollama import ChatOllama
llm1 = ChatOllama(model="qwen3:1.7b")  # or "llama3", etc.

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
hf_token = os.getenv("HF_TOKEN")
model = HuggingFaceEndpoint(
            repo_id="meta-llama/Llama-3.3-70B-Instruct",
            task="text-generation",
            huggingfacehub_api_token=hf_token
        )

llm2 =  ChatHuggingFace(llm=model, temperature=0.5 )
print(llm2.invoke("Hello").content)

Hello. How can I assist you today?


In [3]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(f"Available tables : {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Available tables : ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [4]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db = db, llm = llm2)
tools = toolkit.get_tools()

for tool in tools:
  print(f"{tool.name} : {tool.description}\n")

sql_db_query : Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema : Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables : Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker : Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



## Agent

In [5]:
from typing import Annotated, Sequence, TypedDict
from langchain_core.messages import BaseMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode, tools_condition

system = """
You are a skilled SQL assistant responsible for answering user questions by interacting with a relational database.
You must strictly follow this step-by-step workflow to get the correct answer:

1. Discover Tables: Call `sql_db_list_tables` to see what tables are available.
2. Understand Schema: Call `sql_db_schema` with relevant table names to inspect columns, types, and foreign keys.
3. Validate SQL: Generate a SQL query and ALWAYS call `sql_db_query_checker` to validate it.
4. Execute SQL: Call `sql_db_query` to run the validated query and retrieve the actual data.
5. Final Answer: Formulate your final response based ONLY on the data returned by `sql_db_query`.

CRITICAL RULES:
- NEVER guess or hallucinate any database results.
- Running `sql_db_query_checker` only validates syntax. It does NOT run the query or return database records.
- You MUST execute the SQL query using `sql_db_query` before providing a final answer.
- If you have not successfully run `sql_db_query`, you do NOT have the data required to answer. Do not guess the records.
- Never perform destructive operations (DROP, DELETE, UPDATE, etc.).
- Ensure your query correctly joins tables and uses appropriate aggregations/filters.
"""



In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def call_model(state: AgentState):
    messages = state["messages"]
    response = llm2.bind_tools(tools).invoke([("system", system)] + list(messages))
    return {"messages": [response]}

workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", ToolNode(tools))


workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent",
    tools_condition,
)
workflow.add_edge("tools", "agent")

agent = workflow.compile()

In [ ]:
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [8]:
question = "Which genre on average has the longest tracks?"

response = agent.invoke(
    {"messages": [("user", question)]}
)
print(response["messages"][-1].content)


The genre with the longest tracks on average is Sci Fi & Fantasy, with an average track length of 2911783.0384615385 milliseconds.


In [9]:
question = "What is the schema of the db?"

response = agent.invoke(
    {"messages": [("user", question)]}
)
print(response["messages"][-1].content)


The schema of the database includes the following tables:
- Album: This table contains information about music albums, including the album ID, title, and artist ID.
- Artist: This table contains information about music artists, including the artist ID and name.
- Customer: This table contains information about customers, including customer ID, first and last name, company, address, city, state, country, postal code, phone, fax, email, and support representative ID.
- Employee: This table contains information about employees, including employee ID, last and first name, title, reports to, birth date, hire date, address, city, state, country, postal code, phone, fax, and email.
- Genre: This table contains information about music genres, including the genre ID and name.
- Invoice: This table contains information about invoices, including the invoice ID, customer ID, invoice date, billing address, billing city, billing state, billing country, billing postal code, and total.
- InvoiceLine: 